# 12.5 · GPT 与解码器模型 / GPT & Decoder Models ⭐⭐⭐

> **课程定位 / Where this fits**
> 第 5 课，**Part 12**。Transformer 第二种用法：**解码器**——也是 ChatGPT、GPT-4 等所有生成式大模型的架构。
> Lesson 5, **Part 12**. The second Transformer use: the **decoder** — the architecture of ChatGPT, GPT-4, and all generative LLMs.
>
> BERT(12.4)双向、擅理解。**GPT** 走另一条路：**只用解码器 + 因果(单向)自注意力**，做**自回归生成**——给定前文，逐个预测下一个 token，把预测拼回去再预测下一个……如此"接龙"出整段文本。规模够大时还涌现出**上下文学习(in-context learning)** 等惊人能力。本课**从零搭一个字符级 GPT**，在真实文本(简·奥斯汀小说)上训练，让它**学会生成英文**，并探索**采样温度**对生成的影响。
> BERT (12.4) is bidirectional, great at understanding. **GPT** takes the other path: **decoder-only + causal (unidirectional) self-attention** for **autoregressive generation** — given a prefix, predict the next token, append it, predict again… "continuing the story" into full text. At scale, abilities like **in-context learning** emerge. We **build a char-level GPT from scratch**, train it on real text (Jane Austen), have it **learn to generate English**, and explore how **sampling temperature** shapes output.
>
> 💼 **实战/面试视角**：GPT 是 LLM 面试**核心**——"自回归生成 / 因果掩码 / 解码策略(贪心/采样/温度/top-k/top-p) / GPT vs BERT" 几乎必考。
> 💼 **Practical/interview angle:** GPT is **central** to LLM interviews — autoregressive generation, causal masking, decoding strategies (greedy/sampling/temperature/top-k/top-p), GPT vs BERT.

> 📐 **符号约定 / Notation**
> - 自回归 —— 用已生成的前文预测下一个 token / predict next token from generated prefix
> - 因果掩码 —— 屏蔽"未来"位置, 每个 token 只能看左边 / causal mask: each token sees only the left
> - 温度 $\tau$ —— 调节采样随机性 / sampling temperature

> 💡 **面试相关 / Interview-relevant**
> - "自回归语言模型/因果掩码"（出镜率 ★★★★★）
> - "解码策略: 贪心/温度/top-k/top-p(nucleus)"（★★★★★）
> - "GPT vs BERT(单向生成 vs 双向理解)"（★★★★★）
> - "温度的作用"（★★★★）
> - "in-context learning / 涌现能力"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解 GPT = 解码器 + 因果注意力 + 自回归生成。
   Understand GPT = decoder + causal attention + autoregressive generation.
2. **从零实现因果掩码**并可视化。
   Implement causal masking from scratch and visualize.
3. **从零搭字符级 GPT** 并在真实文本上训练。
   Build a char-level GPT from scratch and train on real text.
4. **生成文本**，掌握采样温度等解码策略。
   Generate text; master temperature and decoding strategies.

## 目录 / TOC
1. [GPT：自回归生成 ⭐](#1)
2. [因果掩码：只能看左边（从零）⭐](#2)
3. [从零搭字符级 GPT 并训练 ⭐](#3)
4. [文本生成与解码策略 + 小结 ⭐](#4)


<a id="1"></a>
## 1. GPT：自回归生成 ⭐ / GPT: Autoregressive Generation

**GPT = Generative Pre-trained Transformer**。训练目标极简：**预测下一个 token**(给定前文 $x_1..x_{t-1}$，预测 $x_t$)。这叫**因果语言模型(causal LM)** 或**自回归(autoregressive)**。
**GPT = Generative Pre-trained Transformer.** The training objective is dead simple: **predict the next token** (given prefix $x_1..x_{t-1}$, predict $x_t$). This is a **causal LM** / **autoregressive** model.

**生成("接龙")过程**：给一个起始文本 → 模型输出下一个 token 的概率分布 → 采一个 token → 拼到末尾 → 用新序列再预测下一个 → 重复。就这样一个一个 token 地"写"出整段文本。ChatGPT 本质就是这个循环(只是模型巨大、且经过对齐训练)。
**Generation ("continuation"):** give a prompt → the model outputs a probability distribution over the next token → sample one → append → predict the next from the new sequence → repeat. Token by token, it "writes" full text. ChatGPT is essentially this loop (just a huge, aligned model).

**关键(面试)**：训练时可以**并行**算所有位置的"下一个词"预测(每个位置都有标签=它的下一个词，一次前向全算)；但**生成时必须串行**(下一个 token 依赖已生成的)。这是 LLM 推理慢的根源(12.16 讲优化)。
**Key (interview):** training is **parallel** — every position has a label (its next token), computed in one forward pass; but **generation is serial** (each token depends on prior ones). This is why LLM inference is slow (optimized in 12.16).


<a id="2"></a>
## 2. 因果掩码：只能看左边（从零）⭐ / Causal Mask: See Only the Left

GPT 和 BERT 的**架构几乎一样**(都是 Transformer 块)，唯一关键区别：**因果掩码(causal mask)**。
GPT and BERT have **nearly identical architectures** (Transformer blocks); the one key difference is the **causal mask**.

预测第 $t$ 个词时，模型**绝不能看到第 $t$ 个及之后**的词(否则就是看着答案抄)。因果掩码在自注意力的分数矩阵上，把每个位置**对未来位置的注意力**设成 $-\infty$(softmax 后变 0)。于是每个 token **只能注意到自己和左边**。
When predicting token $t$, the model **must not see token $t$ or later** (else it's copying the answer). The causal mask sets each position's attention to **future positions** to $-\infty$ (→ 0 after softmax). So each token **attends only to itself and the left**.

这正是 **GPT(单向) vs BERT(双向)** 的本质区别：BERT 无掩码(双向理解)，GPT 加因果掩码(单向生成)。
This is the essence of **GPT (unidirectional) vs BERT (bidirectional)**: BERT has no mask (bidirectional understanding), GPT adds the causal mask (unidirectional generation).


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, math, time
import torch, torch.nn as nn, torch.nn.functional as F
sns.set_theme(style="white")
torch.manual_seed(0)

def causal_mask(T):
    """下三角矩阵: 位置 i 只能看 j<=i / lower-triangular: position i can attend to j<=i."""
    return torch.tril(torch.ones(T, T))                   # tril = 保留下三角(含对角) / keep lower triangle

M = causal_mask(8)
fig, ax = plt.subplots(figsize=(5, 4.5))
sns.heatmap(M, annot=True, fmt=".0f", cmap="Greens", cbar=False,
            xticklabels=[f"t{j}" for j in range(8)], yticklabels=[f"t{i}" for i in range(8)], ax=ax)
ax.set_xlabel("被关注位置 (Key)"); ax.set_ylabel("查询位置 (Query)")
ax.set_title("因果掩码: 1=可关注, 0=屏蔽; 每个位置只能看自己和左边(下三角)")
plt.tight_layout(); plt.show()
print("生成第t个词时屏蔽t及之后(置-∞→softmax后0) → 每个token只看左边 → 单向'因果'")
print("GPT(因果掩码,单向生成) vs BERT(无掩码,双向理解): 架构同, 掩码不同")


<a id="3"></a>
## 3. 从零搭字符级 GPT 并训练 ⭐ / Char-Level GPT From Scratch

我们搭一个**字符级 GPT**：词表就是文本里出现的所有字符，模型学"给定前面的字符，下一个字符是什么"。在**简·奥斯汀的小说**(nltk 自带)上训练。字符级最简单(不需要分词器，12.7 再讲分词)，却足以让模型学会**单词拼写、空格、标点、甚至文风**。
We build a **char-level GPT**: the vocabulary is all characters in the text, and the model learns "given previous chars, what's the next char." We train on **Jane Austen** (in nltk). Char-level is simplest (no tokenizer needed; tokenization is 12.7) yet enough to learn **spelling, spaces, punctuation, even style**.


In [ ]:
import nltk; nltk.download("gutenberg", quiet=True)
from nltk.corpus import gutenberg
text = gutenberg.raw("austen-sense.txt")[:200000]          # 取 20 万字符 / 200k chars
chars = sorted(set(text)); V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text]); n = int(0.9*len(data)); train_d, val_d = data[:n], data[n:]
CTX = 64                                                    # 上下文长度(模型一次看多少字符) / context length
print(f"语料 {len(text)} 字符, 字符表大小 {V}, 上下文长度 {CTX}")

def get_batch(d, B=64):
    ix = torch.randint(len(d)-CTX-1, (B,))
    x = torch.stack([d[i:i+CTX] for i in ix])
    y = torch.stack([d[i+1:i+CTX+1] for i in ix])           # 标签=每个位置的下一个字符 / next-char labels
    return x, y

class CausalMHA(nn.Module):
    def __init__(s, D, H): super().__init__(); s.H=H; s.d=D//H; s.qkv=nn.Linear(D,3*D); s.o=nn.Linear(D,D)
    def forward(s, x, mask):
        B,T,D = x.shape; qkv = s.qkv(x).reshape(B,T,3,s.H,s.d).permute(2,0,3,1,4); q,k,v = qkv
        sc = q @ k.transpose(-2,-1) / math.sqrt(s.d)
        sc = sc.masked_fill(mask[:T,:T]==0, float("-inf"))  # 因果掩码: 屏蔽未来 / causal mask
        a = F.softmax(sc, -1); o = (a@v).transpose(1,2).reshape(B,T,D); return s.o(o)
class Block(nn.Module):
    def __init__(s, D, H): super().__init__(); s.a=CausalMHA(D,H); s.n1=nn.LayerNorm(D); s.n2=nn.LayerNorm(D); s.ff=nn.Sequential(nn.Linear(D,4*D),nn.GELU(),nn.Linear(4*D,D))
    def forward(s, x, m): x=x+s.a(s.n1(x),m); return x+s.ff(s.n2(x))
class GPT(nn.Module):
    def __init__(s, V, D=128, H=4, L=3, ctx=CTX):
        super().__init__(); s.tok=nn.Embedding(V,D); s.pos=nn.Embedding(ctx,D)
        s.blocks=nn.ModuleList([Block(D,H) for _ in range(L)]); s.ln=nn.LayerNorm(D); s.head=nn.Linear(D,V); s.ctx=ctx
        s.register_buffer("mask", torch.tril(torch.ones(ctx,ctx)))
    def forward(s, x):
        T=x.size(1); h = s.tok(x) + s.pos(torch.arange(T))
        for b in s.blocks: h = b(h, s.mask)
        return s.head(s.ln(h))
    @torch.no_grad()
    def generate(s, idx, n_new, temp=0.8, top_k=None):
        for _ in range(n_new):
            logits = s(idx[:, -s.ctx:])[:, -1] / temp       # 最后位置的下一字符分布 / next-char logits
            if top_k:                                       # top-k 采样: 只在概率最高的k个里采 / top-k sampling
                v, _ = torch.topk(logits, top_k); logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, -1)
            nxt = torch.multinomial(probs, 1)               # 按概率采样下一个字符 / sample next char
            idx = torch.cat([idx, nxt], dim=1)              # 拼回去, 继续生成 / append and continue
        return idx

torch.manual_seed(0); model = GPT(V); opt = torch.optim.AdamW(model.parameters(), 3e-3)
print(f"GPT 参数量 {sum(p.numel() for p in model.parameters()):,}")
losses = []; t0 = time.time()
for step in range(1500):
    x, y = get_batch(train_d); logits = model(x)
    loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
fig, ax = plt.subplots(figsize=(7,3)); ax.plot(losses); ax.set_xlabel("训练步"); ax.set_ylabel("交叉熵损失")
ax.set_title(f"字符级 GPT 训练 ({time.time()-t0:.0f}s)"); plt.tight_layout(); plt.show()
print(f"训练完成, 损失 {losses[0]:.2f} → {losses[-1]:.2f} (损失=预测下一字符的不确定度)")


<a id="4"></a>
## 4. 文本生成与解码策略 + 小结 ⭐ / Generation & Decoding Strategies

训练好后，从一个起始字符开始**自回归生成**。每步模型给出下一个字符的概率分布，怎么"选"这个字符就是**解码策略**(面试高频)：
After training, **autoregressively generate** from a starting character. Each step yields a next-char distribution; how we "pick" is the **decoding strategy** (high-frequency interview):
- **贪心(greedy)**：每步选概率最高的。确定、但易重复、单调。
  **Greedy:** always take the highest-probability token. Deterministic but repetitive/dull.
- **温度采样(temperature)**：按概率随机采，温度 $\tau$ 调随机性：$\tau$ 小→更保守(接近贪心)，$\tau$ 大→更随机/有创意(但易胡言)。
  **Temperature sampling:** sample by probability; low $\tau$ → conservative (near greedy), high $\tau$ → random/creative (but more gibberish).
- **top-k / top-p(nucleus)**：只在概率最高的 k 个 / 累积概率达 p 的若干 token 里采样，避免采到长尾垃圾词。
  **top-k / top-p (nucleus):** sample only among the top-k / smallest set summing to prob p, avoiding long-tail junk.

下面用不同温度生成，直观感受温度的影响。
Let's generate at different temperatures to feel the effect.


In [ ]:
start = torch.tensor([[stoi["T"]]])                       # 从字符 'T' 开始 / start from 'T'
for temp in [0.4, 0.8, 1.2]:
    out = model.generate(start, 220, temp=temp, top_k=20)[0].tolist()
    txt = "".join(itos[i] for i in out).replace("\n", " ")
    print(f"=== 温度 τ={temp} ===")
    print(txt[:200])
    print()
print("观察: τ低→更保守/重复但更通顺; τ高→更多样/有创意但更易出错")
print("字符级GPT(几十秒训练)已学会: 拼出真实英文单词+空格标点+奥斯汀文风 → 自回归生成的威力")
print("真实GPT: 同样的'预测下一个token'目标, 但用子词分词(12.7)+海量数据+巨大模型 → 涌现出对话/推理能力")


```
GPT: 解码器only + 因果(单向)自注意力; 目标=预测下一个token(自回归/因果LM)
生成: 给前文→预测下一token分布→采样→拼回→重复; ChatGPT 本质就是这个循环
因果掩码: 屏蔽未来位置(分数置-∞→softmax后0); 每token只看左边; 这是GPT vs BERT唯一关键区别
训练并行(每位置都有下一词标签一次算完) vs 生成串行(逐token依赖) → 推理慢(12.16优化)
解码策略: 贪心(确定/重复) / 温度(调随机性) / top-k / top-p(nucleus, 避免长尾垃圾)
温度τ: 低=保守通顺, 高=多样易错; 实战常 τ≈0.7 + top-p≈0.9
规模: GPT-1→2→3→4 越大越强; 够大时涌现 in-context learning(看几个例子就会做新任务)
```

### 💡 面试速查 / Interview cheat-sheet
1. **自回归**: 预测下一个token, 逐个生成并拼回; 训练并行/生成串行。
   Autoregressive: predict next token, generate & append; parallel train / serial generate.
2. **因果掩码**: 屏蔽未来→每token只看左; GPT vs BERT 的关键区别。
   Causal mask: block the future → left-only; the key GPT-vs-BERT difference.
3. **解码策略**: 贪心/温度/top-k/top-p; 控制确定性vs多样性。
   Decoding: greedy/temperature/top-k/top-p; control determinism vs diversity.
4. **温度**: 低=保守, 高=有创意但易胡言; 常用 0.7+top-p 0.9。
   Temperature: low=safe, high=creative but error-prone; common 0.7 + top-p 0.9.
5. **GPT vs BERT**: 单向生成 vs 双向理解; 架构同, 掩码不同。
   GPT vs BERT: unidirectional generation vs bidirectional understanding; same arch, different mask.

### 下一节 / Next
**12.6 T5 与编码器-解码器**——Transformer 第三种用法: 编码器+解码器一起用, 把**所有任务统一成"文本到文本"**(输入文本→输出文本)。这是翻译、摘要等"转换"类任务的利器, 也是一种优雅的统一框架。
**12.6 T5 & Encoder-Decoder** — the third Transformer use: encoder + decoder together, framing **every task as "text-to-text"** (input text → output text). Powerful for transduction (translation, summarization) and an elegant unified framework.
